# Exploração SQL e separação temporal

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [ ]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="03_exploracao_preparacao_dados"


1. Mostrar SQL versionado, recorte, fonte e qualidade. Esperado: dados sintéticos lidos por conexão SQL fechada pelo módulo.

In [ ]:
from src.data.journey import extract,prepare,QUERY,KEYS
frame,snapshot=extract(cfg,snapshot_id=str(params.snapshot_id) if params.snapshot_id else None)
display(Markdown("```sql\n"+QUERY.read_text()+"\n```"));display(snapshot)
display(frame.dtypes.astype(str));display(frame.head(12));display(frame.describe(include="all"))
display(frame.isna().sum());assert not frame.duplicated(KEYS+["semester_index"]).any()
evidence.update(snapshot=snapshot,rows=len(frame),nulls=frame.isna().sum().to_dict())


2. Evolução e split temporal. Transformações serão ajustadas somente no treino.

In [ ]:
from IPython.display import SVG
from src.workbench.exploration import series_svg
display(SVG(series_svg(frame)))
train,evaluation,future=prepare(frame,snapshot)
split={"train":len(train),"evaluation":len(evaluation),"future":len(future),"train_max":int(train.semester_index.max()),"evaluation_min":int(evaluation.semester_index.min()),"future_min":int(future.semester_index.min())}
display(split);evidence["split"]=split


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [ ]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))
